# 01. The Wild West: Unstructured Extraction Failures
**Design Pattern Context:** The LLM Wild West vs. Production Integration  
**Model:** `openai/gpt-oss-120b` on Groq Cloud  

This notebook demonstrates what happens when downstream application services rely on raw natural language LLM completions without strict grammar or schema gating.

In [16]:
!pip install -q groq pydantic

import os
import json
from groq import Groq

if "GROQ_API_KEY" not in os.environ:
    os.environ["GROQ_API_KEY"] = "your-groq-api-key-here"

client = Groq(api_key=os.environ["GROQ_API_KEY"])
MODEL_ID = "openai/gpt-oss-120b"

## 1. Scenario: Ambiguous Natural Language Input

In [17]:
user_prompt = "Book me an aisle seat to Tokyo next Friday under 900 bucks, make sure it's a vegan meal. Name is Jaichand."

system_prompt = """
You are a flight reservation assistant. Extract the flight details from the user prompt into raw JSON.
Include the following keys:
- passenger_name (string)
- destination (string)
- seat_preference (must be: aisle, aisle_quiet, window, middle)
- budget_limit_usd (integer)
- meal (must be: vegan, standard, none)
Return ONLY JSON. Do not add explanations.
"""

response = client.chat.completions.create(
    model=MODEL_ID,
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ],
    temperature=0.7
)

raw_output = response.choices[0].message.content
print("--- RAW MODEL OUTPUT ---")
print(raw_output)

--- RAW MODEL OUTPUT ---
{
  "passenger_name": "Jaichand",
  "destination": "Tokyo",
  "seat_preference": "aisle",
  "budget_limit_usd": 900,
  "meal": "vegan"
}


## 2. Downstream Transaction Pipeline & Realistic Breakages

In [18]:
def process_booking_downstream(booking_data: dict):
    """
    Simulates database insertions and payment processing.
    """
    print(f"\n[Payment Gateway] Reserving budget limit: ${booking_data['budget_limit_usd']:.2f}")
    
    allowed_meals = {'vegan', 'standard', 'none'}
    if booking_data['meal'] not in allowed_meals:
        raise ValueError(f"Constraint Violation: Meal '{booking_data['meal']}' not supported by catering service!")
        
    query = (
        f"INSERT INTO bookings (name, destination, budget, meal) "
        f"VALUES ('{booking_data['passenger_name']}', '{booking_data['destination']}', "
        f"{int(booking_data['budget_limit_usd'])}, '{booking_data['meal']}');"
    )
    print(f"[Database Engine] Executing SQL: {query}")
    return True

In [19]:
# Breakage 1: Markdown Syntax Fences crashing json.loads()
sample_markdown_fence = f"```json\n{raw_output.strip('`').replace('json', '').strip()}\n```"
print("Testing parsing of markdown wrapped output:")
try:
    data = json.loads(sample_markdown_fence)
except json.JSONDecodeError as e:
    print(f"FAILED: json.loads() threw JSONDecodeError:\n-> {e}")

Testing parsing of markdown wrapped output:
FAILED: json.loads() threw JSONDecodeError:
-> Expecting value: line 1 column 1 (char 0)


In [20]:
# Breakage 2: Field Key Drift causing Missing Key crashes
drifted_payload = {
    "passenger_full_name": "Jaichand",
    "target_airport": "Tokyo",
    "seat": "aisle",
    "budget_limit_usd": 900,
    "meal": "vegan"
}

try:
    print("Testing payload with drifted keys:")
    process_booking_downstream(drifted_payload)
except KeyError as e:
    print(f"CRITICAL API FAILURE: Missing expected key {e}")

Testing payload with drifted keys:

[Payment Gateway] Reserving budget limit: $900.00
CRITICAL API FAILURE: Missing expected key 'passenger_name'


In [21]:
# Breakage 3: Type Poisoning & Enum Breaches
type_polluted_payload = {
    "passenger_name": "Jaichand",
    "destination": "Tokyo",
    "seat_preference": "aisle",
    "budget_limit_usd": "under 900",
    "meal": "strict vegetarian"
}

try:
    print("Testing type-polluted payload:")
    process_booking_downstream(type_polluted_payload)
except (TypeError, ValueError) as e:
    print(f"TRANSACTION ROLLED BACK: Data integrity check failed:\n-> {e}")

Testing type-polluted payload:
TRANSACTION ROLLED BACK: Data integrity check failed:
-> Unknown format code 'f' for object of type 'str'
